# SNR-Aware Low-light Image Enhancement — Implementation / 구현

**Paper**: Xu, X., Wang, R., Fu, C.-W., & Jia, J. (2022). SNR-Aware Low-light Image Enhancement. *CVPR*, 17714–17724. DOI: 10.1109/CVPR52688.2022.01719

This notebook implements a small-scale version of the paper's core ideas:
1. Per-pixel **SNR map** estimation from a single low-light image (Eq. 2)
2. A two-branch network: a **short-range** convolutional branch and a **long-range** non-local (transformer-style) branch
3. **SNR-guided fusion** that linearly blends the two branch features using the normalized SNR map (Eq. 3)
4. A small training loop (≤100 iters) on synthetically darkened patches and a final summary table comparing single-branch vs SNR-aware fusion

이 노트북은 논문의 핵심 아이디어를 소규모로 구현합니다: (1) 단일 영상에서의 픽셀별 SNR 맵 추정 (Eq. 2), (2) 단거리 CNN 분기와 장거리 비국소 분기로 구성된 두 분기 네트워크, (3) 정규화된 SNR 맵을 가중치로 하는 SNR-guided fusion (Eq. 3), (4) 합성 압널 패치에서의 소규모 학습(≤100 iter) 및 단일 분기 vs SNR-aware fusion 비교 표.

## Setup / 설정

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Part 1: SNR Map Estimation / SNR 맵 추정 (Eq. 2)

The paper estimates per-pixel SNR from a single image without any learning:

$$\hat I_g = \text{denoise}(I_g), \quad N = |I_g - \hat I_g|, \quad S = \hat I_g \,/\, N$$

We use a local mean filter as the denoiser (the default in the paper, chosen for speed; Sec. 4.4 shows non-local means and BM3D give nearly identical performance).

논문은 학습 없이 단일 영상에서 픽셀별 SNR을 추정한다. denoise 연산자는 local mean filter를 사용 (논문 Sec. 4.4은 NLM, BM3D 모두 거의 동등한 성능을 보임을 확인).

In [ ]:
def rgb_to_gray(image: Tensor) -> Tensor:
    """Convert an RGB tensor to grayscale using ITU-R BT.601 weights.

    Args:
        image: Tensor of shape (B, 3, H, W) with values in [0, 1].

    Returns:
        Grayscale tensor of shape (B, 1, H, W).
    """
    weights = torch.tensor([0.299, 0.587, 0.114], device=image.device).view(1, 3, 1, 1)
    return (image * weights).sum(dim=1, keepdim=True)


def local_mean(gray: Tensor, kernel_size: int = 5) -> Tensor:
    """Box-filter local mean denoiser (the paper's default fast denoiser).

    Args:
        gray: Tensor of shape (B, 1, H, W).
        kernel_size: Odd integer giving the box-filter width.

    Returns:
        Locally-averaged tensor of the same shape.
    """
    padding = kernel_size // 2
    kernel = torch.ones(1, 1, kernel_size, kernel_size, device=gray.device) / (
        kernel_size * kernel_size
    )
    return F.conv2d(gray, kernel, padding=padding)


def compute_snr_map(image: Tensor, kernel_size: int = 5, eps: float = 1e-4) -> Tensor:
    """Compute the per-pixel SNR map S = denoise(I_g) / |I_g - denoise(I_g)| (Eq. 2).

    The SNR map is then min-max normalized to [0, 1], matching the paper's S' that is
    used as a soft attention weight in fusion (Eq. 3).

    Args:
        image: Low-light RGB tensor of shape (B, 3, H, W) with values in [0, 1].
        kernel_size: Width of the local-mean denoiser.
        eps: Numerical floor for the noise estimate to avoid division by zero.

    Returns:
        Normalized SNR map S' of shape (B, 1, H, W) with values in [0, 1].
    """
    gray = rgb_to_gray(image)
    denoised = local_mean(gray, kernel_size=kernel_size)
    noise = (gray - denoised).abs().clamp(min=eps)
    snr = denoised / noise

    # Per-image min-max normalization to [0, 1].
    flat = snr.flatten(start_dim=1)
    s_min = flat.min(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    s_max = flat.max(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    snr_normalized = (snr - s_min) / (s_max - s_min + eps)
    return snr_normalized.clamp(0.0, 1.0)

### Visual sanity check / 시각 검증

Build a synthetic low-light image by gamma-darkening a clean test image, then compare clean / dark / SNR map.

깨끗한 테스트 영상을 감마 보정으로 어둡게 만들고 우리 SNR 맵이 넓은 동적 범위를 포착하는지 확인.

In [ ]:
def make_gradient_scene(height: int = 64, width: int = 64) -> np.ndarray:
    """Create a synthetic ground-truth scene with smooth gradient + textured stripes.

    Args:
        height: Output image height in pixels.
        width: Output image width in pixels.

    Returns:
        NumPy array of shape (height, width, 3) with values in [0, 1].
    """
    yy, xx = np.meshgrid(
        np.linspace(0.1, 1.0, height), np.linspace(0.1, 1.0, width), indexing="ij"
    )
    base = (yy + xx) / 2.0
    stripes = 0.15 * np.sin(2 * np.pi * 6 * xx) * np.cos(2 * np.pi * 4 * yy)
    gray = np.clip(base + stripes, 0.05, 1.0)
    rgb = np.stack(
        [gray, np.clip(gray * 0.95 + 0.03, 0, 1), np.clip(gray * 0.9 + 0.05, 0, 1)],
        axis=-1,
    )
    return rgb.astype(np.float32)


def darken(image: np.ndarray, gamma: float = 4.0, noise_std: float = 0.04) -> np.ndarray:
    """Synthesize a low-light, noisy version of a clean image.

    Args:
        image: Clean image of shape (H, W, 3), values in [0, 1].
        gamma: Gamma exponent (>1 darkens).
        noise_std: Standard deviation of additive Gaussian noise.

    Returns:
        Darkened noisy image of the same shape.
    """
    dark = image ** gamma
    noise = np.random.normal(0.0, noise_std, size=image.shape).astype(np.float32)
    return np.clip(dark + noise, 0.0, 1.0)


clean_np = make_gradient_scene(64, 64)
dark_np = darken(clean_np, gamma=4.0, noise_std=0.05)

clean_t = torch.from_numpy(clean_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
dark_t = torch.from_numpy(dark_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
snr_t = compute_snr_map(dark_t)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(clean_np)
axes[0].set_title("Clean / GT")
axes[1].imshow(dark_np)
axes[1].set_title("Low-light input")
im = axes[2].imshow(snr_t[0, 0].cpu().numpy(), cmap="magma")
axes[2].set_title("Normalized SNR map S'")
plt.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 2: Two-branch SNR-aware Network / 두 분기 SNR-aware 네트워크

We build a small encoder → [short-range conv branch + long-range non-local branch] → SNR-guided fusion (Eq. 3) → decoder → residual to input. The non-local branch is a lightweight stand-in for the paper's transformer (a single self-attention block over flattened spatial tokens), which captures the same long-range coupling at small scale.

소규모 구조: encoder → [단거리 conv 분기 + 장거리 비국소 분기] → SNR-guided fusion → decoder → 잔차 연결. 비국소 분기는 논문의 패치 트랜스포머를 장난감 규모의 단일 self-attention 블록으로 대체.

In [ ]:
class ConvBlock(nn.Module):
    """Conv -> LeakyReLU -> Conv with residual connection (paper's short-range unit)."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: Tensor) -> Tensor:
        """Forward pass.

        Args:
            x: Input feature tensor of shape (B, C, H, W).

        Returns:
            Output feature tensor of shape (B, C, H, W).
        """
        return x + self.conv2(self.act(self.conv1(x)))


class ShortRangeBranch(nn.Module):
    """Stack of residual conv blocks; captures local spatial detail."""

    def __init__(self, channels: int, num_blocks: int = 3) -> None:
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels) for _ in range(num_blocks)])

    def forward(self, x: Tensor) -> Tensor:
        """Apply the conv stack."""
        return self.blocks(x)


class LongRangeBranch(nn.Module):
    """Single-block self-attention over flattened spatial tokens.

    A lightweight surrogate for the paper's patch-wise transformer: each spatial
    location is one token, and we run multi-head self-attention across them.
    """

    def __init__(self, channels: int, num_heads: int = 4) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.GELU(),
            nn.Linear(channels * 2, channels),
        )
        self.norm2 = nn.LayerNorm(channels)

    def forward(self, x: Tensor) -> Tensor:
        """Apply self-attention + FFN with pre-norm and residual connections.

        Args:
            x: Feature tensor of shape (B, C, H, W).

        Returns:
            Tensor of shape (B, C, H, W) with the same channels.
        """
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)  # (B, HW, C)
        attended, _ = self.attn(self.norm(tokens), self.norm(tokens), self.norm(tokens))
        tokens = tokens + attended
        tokens = tokens + self.ffn(self.norm2(tokens))
        return tokens.transpose(1, 2).reshape(b, c, h, w)


@dataclass
class SNRNetConfig:
    """Config for the small SNR-aware enhancement network."""

    in_channels: int = 3
    base_channels: int = 32
    num_short_blocks: int = 3
    num_heads: int = 4
    use_snr_fusion: bool = True


class SNRAwareNet(nn.Module):
    """Encoder + two branches + SNR-guided fusion + decoder.

    When `use_snr_fusion=False` the model collapses to a single-branch
    (short-range conv only) baseline so we can ablate the fusion's contribution.
    """

    def __init__(self, cfg: SNRNetConfig) -> None:
        super().__init__()
        self.cfg = cfg
        c = cfg.base_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(cfg.in_channels, c, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.short_branch = ShortRangeBranch(c, num_blocks=cfg.num_short_blocks)
        if cfg.use_snr_fusion:
            self.long_branch = LongRangeBranch(c, num_heads=cfg.num_heads)
        self.decoder = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(c, cfg.in_channels, 3, padding=1),
        )

    def forward(self, image: Tensor, snr_map: Tensor) -> Tensor:
        """Run the network and return the enhanced image.

        Args:
            image: Low-light RGB tensor (B, 3, H, W) in [0, 1].
            snr_map: Normalized SNR map (B, 1, H, W) in [0, 1].

        Returns:
            Enhanced image tensor (B, 3, H, W). Output is image + residual.
        """
        feat = self.encoder(image)
        short_feat = self.short_branch(feat)
        if self.cfg.use_snr_fusion:
            long_feat = self.long_branch(feat)
            # Eq. 3 fusion: high-SNR -> short-range, low-SNR -> long-range.
            fused = short_feat * snr_map + long_feat * (1.0 - snr_map)
        else:
            fused = short_feat
        residual = self.decoder(fused)
        return (image + residual).clamp(0.0, 1.0)

## Part 3: Synthetic Dataset & Training Loop / 합성 데이터셋 및 학습 루프

Train each variant for at most 100 iterations on randomly generated 64×64 patches that we corrupt with gamma + Gaussian noise. We compute PSNR over a held-out test patch.

랜덤 생성한 64×64 패치에 감마 + 가우시안 노이즈를 가한 다음 최대 100 iter로 학습. 동일한 테스트 패치에서 PSNR 측정.

In [ ]:
def random_low_light_pair(
    height: int = 64,
    width: int = 64,
    gamma_range: Tuple[float, float] = (3.0, 5.0),
    noise_range: Tuple[float, float] = (0.03, 0.07),
) -> Tuple[Tensor, Tensor]:
    """Sample a (low-light, ground-truth) image pair.

    Args:
        height: Patch height.
        width: Patch width.
        gamma_range: Range to sample gamma exponent from (uniform).
        noise_range: Range for Gaussian noise std (uniform).

    Returns:
        Tuple `(low, gt)` of tensors with shape (1, 3, H, W).
    """
    clean = make_gradient_scene(height, width)
    # Add randomized colored blobs so the dataset is non-trivial across runs.
    yy, xx = np.meshgrid(np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="ij")
    for _ in range(np.random.randint(1, 4)):
        cy, cx = np.random.uniform(0.2, 0.8, size=2)
        radius = np.random.uniform(0.08, 0.2)
        color = np.random.uniform(0.4, 1.0, size=3).astype(np.float32)
        mask = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * radius ** 2))
        for ch in range(3):
            clean[..., ch] = np.clip(clean[..., ch] + 0.4 * mask * color[ch], 0, 1)

    gamma = float(np.random.uniform(*gamma_range))
    noise_std = float(np.random.uniform(*noise_range))
    low = darken(clean, gamma=gamma, noise_std=noise_std)

    low_t = torch.from_numpy(low).permute(2, 0, 1).unsqueeze(0)
    gt_t = torch.from_numpy(clean).permute(2, 0, 1).unsqueeze(0)
    return low_t, gt_t


def charbonnier_loss(pred: Tensor, target: Tensor, eps: float = 1e-3) -> Tensor:
    """Charbonnier reconstruction loss (Eq. 7 in the paper).

    Args:
        pred: Predicted image (B, C, H, W).
        target: Ground-truth image (B, C, H, W).
        eps: Smoothing constant.

    Returns:
        Scalar loss tensor.
    """
    return torch.sqrt((pred - target) ** 2 + eps ** 2).mean()


def psnr(pred: Tensor, target: Tensor, max_val: float = 1.0) -> float:
    """Compute PSNR in dB between two tensors in [0, 1]."""
    mse = F.mse_loss(pred, target).item()
    if mse <= 1e-12:
        return 99.0
    return 10.0 * float(np.log10((max_val ** 2) / mse))


def train_one_variant(
    use_snr_fusion: bool,
    num_iters: int = 100,
    lr: float = 2e-3,
) -> Tuple[SNRAwareNet, list[float], Tuple[Tensor, Tensor, Tensor, Tensor]]:
    """Train one variant of the network and return its loss curve and a test sample.

    Args:
        use_snr_fusion: If True, train the SNR-aware two-branch model; otherwise the
            single-branch (conv-only) baseline.
        num_iters: Maximum training iterations.
        lr: Adam learning rate.

    Returns:
        Tuple `(model, losses, (low, snr, pred, gt))` for inspection.
    """
    cfg = SNRNetConfig(use_snr_fusion=use_snr_fusion)
    model = SNRAwareNet(cfg).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))

    losses: list[float] = []
    for step in range(num_iters):
        low, gt = random_low_light_pair()
        low, gt = low.to(DEVICE), gt.to(DEVICE)
        snr_map = compute_snr_map(low)

        pred = model(low, snr_map)
        loss = charbonnier_loss(pred, gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    # One held-out test pair with a fixed seed.
    rng_state = np.random.get_state()
    np.random.seed(123)
    low_test, gt_test = random_low_light_pair()
    np.random.set_state(rng_state)
    low_test, gt_test = low_test.to(DEVICE), gt_test.to(DEVICE)
    with torch.no_grad():
        snr_test = compute_snr_map(low_test)
        pred_test = model(low_test, snr_test)
    return model, losses, (low_test, snr_test, pred_test, gt_test)

In [ ]:
print("Training single-branch baseline (no SNR fusion)...")
_, losses_baseline, sample_baseline = train_one_variant(use_snr_fusion=False, num_iters=100)

print("Training SNR-aware two-branch model...")
_, losses_snr, sample_snr = train_one_variant(use_snr_fusion=True, num_iters=100)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_baseline, label="Single-branch (conv only)")
ax.plot(losses_snr, label="SNR-aware fusion")
ax.set_xlabel("Iteration")
ax.set_ylabel("Charbonnier loss")
ax.set_title("Training loss curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Inspect predictions on the held-out test pair / 테스트 쌍에서의 예측 확인

In [ ]:
low_test, snr_test, pred_baseline, gt_test = sample_baseline
_, _, pred_snr, _ = sample_snr

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, im, title in zip(
    axes,
    [low_test, snr_test, pred_baseline, pred_snr, gt_test],
    ["Low-light input", "SNR map S'", "Single-branch", "SNR-aware fusion", "Ground truth"],
):
    arr = im[0].detach().cpu().numpy()
    if arr.shape[0] == 1:
        ax.imshow(arr[0], cmap="magma")
    else:
        ax.imshow(np.transpose(arr, (1, 2, 0)).clip(0, 1))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 4: Summary Table / 요약 표

Compare the two variants on PSNR over the held-out test pair, plus a quick reading of where the SNR fusion routes its information.

두 변종을 테스트 PSNR로 비교하고, SNR fusion이 정보를 어디로 라우팅하는지도 확인.

In [ ]:
psnr_input = psnr(low_test, gt_test)
psnr_baseline = psnr(pred_baseline, gt_test)
psnr_snr = psnr(pred_snr, gt_test)

snr_mean = float(snr_test.mean().item())
snr_low_frac = float((snr_test < 0.3).float().mean().item())
snr_high_frac = float((snr_test > 0.7).float().mean().item())

print("=" * 64)
print(f"{'Variant':<32}{'PSNR (dB)':>12}{'Final loss':>16}")
print("-" * 64)
print(f"{'Low-light input vs GT':<32}{psnr_input:>12.2f}{'-':>16}")
print(f"{'Single-branch (conv only)':<32}{psnr_baseline:>12.2f}{losses_baseline[-1]:>16.4f}")
print(f"{'SNR-aware fusion':<32}{psnr_snr:>12.2f}{losses_snr[-1]:>16.4f}")
print("=" * 64)
print(
    f"SNR map stats on test pair: mean={snr_mean:.3f}, "
    f"fraction<0.3={snr_low_frac:.2f}, fraction>0.7={snr_high_frac:.2f}"
)
print(
    f"PSNR gain from SNR-aware fusion vs single-branch: "
    f"{psnr_snr - psnr_baseline:+.2f} dB"
)

## Concept Summary / 개념 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Per-pixel SNR prior | Local-mean denoiser residual, Eq. 2 | Cheap label-free uncertainty / quality maps used to gate networks |
| Spatial-varying processing | Two-branch fusion via Eq. 3 | Mixture-of-Experts with deterministic gating |
| Noisy-token suppression | SNR-mask softmax, Eq. 6 | Masked self-attention (e.g., Restormer, Retinexformer 2023) |
| Reconstruction loss | Charbonnier + VGG perceptual, Eqs. 7-9 | Standard low-level vision loss combo |